# RNA-seq in the cloud — Google Colab

A minimal, **runnable** RNA-seq pipeline for teaching:

```
FastQC → Trimmomatic → STAR → featureCounts → edgeR → volcano + heatmap
```

This notebook is **self-contained**: upload it to [Colab](https://colab.research.google.com) and run the cells top to bottom — no GitHub repo needed. Along the way you'll see each part of the work in **bash**, **python**, and **R**.

The data is deliberately tiny — a small yeast genome and a simulated set of reads with a built-in difference between two groups (control vs *mip6Δ*) — so the whole pipeline runs in a few minutes on free Colab.

**Two things to know about Colab**
- Step 1 installs `conda` and **restarts the runtime once**. That's normal — just carry on with Step 2 afterwards.
- Colab is **temporary**: files disappear when the session ends, so the last step downloads your results.

## Step 1 — Install conda *(restarts the runtime)*

Run this cell and wait for the *“session crashed/restarted”* message — that's expected. Then continue with Step 2. **Don't re-run this cell.**

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## Step 2 — Install the pipeline tools

Install the command-line tools — STAR, FastQC, Trimmomatic, and featureCounts. The cell prints where each one installed so you can confirm they're ready.

In [ ]:
import condacolab; condacolab.check()   # confirms Step 1 finished

In [ ]:
# condacolab reserves a fixed Python version in the base environment; clearing
# that reservation lets the tool installs below run without a version conflict.
!rm -f /usr/local/conda-meta/pinned

# Install the command-line tools used by the pipeline.
!mamba install -y -n base -c conda-forge -c bioconda star fastqc subread trimmomatic curl

# Confirm they installed — each should print a path.
import shutil
for tool in ["STAR", "fastqc", "trimmomatic", "featureCounts"]:
    print(f"{tool:14s} {shutil.which(tool)}")
assert all(shutil.which(t) for t in ["STAR", "fastqc", "trimmomatic", "featureCounts"]), \
    "A tool above is missing — re-run this cell."

## Step 3 — Reference genome + STAR index *(bash)*

We write a short shell script and run it. It downloads the yeast genome and gene annotation, then builds the STAR index used for alignment.

In [ ]:
%%writefile 00_get_data.sh
#!/usr/bin/env bash
set -euo pipefail

# Download a small yeast reference (~12 Mb) and build the STAR index.
mkdir -p ref && cd ref
REL=110
DNA="https://ftp.ensembl.org/pub/release-${REL}/fasta/saccharomyces_cerevisiae/dna"
GTF="https://ftp.ensembl.org/pub/release-${REL}/gtf/saccharomyces_cerevisiae"

echo ">>> genome FASTA"
curl -sL "${DNA}/Saccharomyces_cerevisiae.R64-1-1.dna.toplevel.fa.gz" -o genome.fa.gz
gunzip -f genome.fa.gz

echo ">>> gene annotation GTF"
curl -sL "${GTF}/Saccharomyces_cerevisiae.R64-1-1.${REL}.gtf.gz" -o genes.gtf.gz
gunzip -f genes.gtf.gz

echo ">>> STAR index"
# genomeSAindexNbases is lowered to 10 because the genome is small.
STAR --runMode genomeGenerate --genomeDir star_index \
  --genomeFastaFiles genome.fa --sjdbGTFfile genes.gtf \
  --sjdbOverhang 100 --genomeSAindexNbases 10 --runThreadN 2

echo ">>> reference ready in ref/, index in ref/star_index/"


In [ ]:
!bash 00_get_data.sh

## Step 4 — Sample sheet + simulated reads *(python)*

First a **sample sheet** (`samples.tsv`: which sample belongs to which group). Then we generate the sequencing reads **in Python, right in the notebook**. There are six samples — three control and three *mip6Δ* — with a built-in expression difference so the analysis has something real to find.

> To use the **real** GEO dataset (GSE135568) instead, download it with `sra-tools`/`pysradb` — see `scripts/get_reads.sh` in the repo.

In [ ]:
%%writefile samples.tsv
sample	gsm	group
ctrl_1	GSM4017984	control
ctrl_2	GSM4017991	control
ctrl_3	GSM4018003	control
mip6_1	GSM4017990	mip6
mip6_2	GSM4017999	mip6
mip6_3	GSM4018006	mip6

In [ ]:
# Simulate a tiny paired-end RNA-seq dataset, using only the Python standard
# library. Reads are drawn from the yeast reference downloaded in Step 3, with a
# built-in expression difference between the two groups (control vs mip6) so the
# differential-expression step finds real hits.
import gzip, os, random, re

READLEN = 90        # length of each read (mate)
FRAGLEN = 180       # fragment size (R1 and R2 sit end-to-end)
N_GENES = 250       # number of genes to simulate reads from
N_UP    = 30        # genes up-regulated in mip6
N_DOWN  = 30        # genes down-regulated in mip6
DE_FOLD = 6         # fold-change for the differentially-expressed genes
QUAL    = "I" * READLEN   # Phred 40 for every base

_COMP = str.maketrans("ACGTNacgtn", "TGCANtgcan")
def revcomp(seq):
    return seq.translate(_COMP)[::-1]

def read_fasta(path):
    """Return {sequence_name: sequence} from a FASTA file."""
    seqs, name, chunks = {}, None, []
    with open(path) as fh:
        for line in fh:
            if line.startswith(">"):
                if name is not None:
                    seqs[name] = "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        seqs[name] = "".join(chunks)
    return seqs

def read_exons(path):
    """Return the longest exon per gene: {gene_id: (chrom, start, end)}."""
    gene_id = re.compile(r'gene_id "([^"]+)"')
    best = {}
    with open(path) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 9 or f[2] != "exon":
                continue
            m = gene_id.search(f[8])
            if not m:
                continue
            gid, chrom, start, end = m.group(1), f[0], int(f[3]), int(f[4])
            if gid not in best or (end - start) > (best[gid][2] - best[gid][1]):
                best[gid] = (chrom, start, end)
    return best

def read_samples(path):
    """Return [(sample, group), ...] from samples.tsv (skipping the header)."""
    rows = []
    with open(path) as fh:
        next(fh, None)
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3 and parts[0]:
                rows.append((parts[0], parts[2]))
    return rows

def pick_genes(exons, genome):
    """Pick N_GENES whose longest exon is big enough to hold a fragment."""
    chosen = []
    for gid in sorted(exons):
        chrom, start, end = exons[gid]
        if chrom in genome and (end - start + 1) >= FRAGLEN + 10:
            chosen.append(gid)
            if len(chosen) >= N_GENES:
                break
    return chosen

def expected_reads(index, group):
    """Expected number of read pairs for gene number `index` in `group`."""
    base = 50 + (index % 11) * 20              # varies across genes (50..250)
    if index < N_UP:                           # up in mip6
        return base * DE_FOLD if group == "mip6" else base
    if index < N_UP + N_DOWN:                  # down in mip6
        return max(5, base // DE_FOLD) if group == "mip6" else base
    return base                                # not differentially expressed

def pick_fragment(seq, start, end, rng):
    """Pick a FRAGLEN window inside the exon, avoiding Ns where possible."""
    lo, hi = start - 1, end - FRAGLEN
    for _ in range(6):
        frag = seq[rng.randint(lo, hi):][:FRAGLEN]
        if "N" not in frag.upper():
            return frag
    return frag

genome  = read_fasta("ref/genome.fa")
exons   = read_exons("ref/genes.gtf")
genes   = pick_genes(exons, genome)
samples = read_samples("samples.tsv")
os.makedirs("data", exist_ok=True)
print(f">>> simulating {len(genes)} genes across {len(samples)} samples")

for sample_i, (sample, group) in enumerate(samples):
    rng = random.Random(1000 + sample_i)       # reproducible per sample
    total = 0
    with gzip.open(f"data/{sample}_R1.fastq.gz", "wt") as r1_out, \
         gzip.open(f"data/{sample}_R2.fastq.gz", "wt") as r2_out:
        for gene_i, gid in enumerate(genes):
            chrom, start, end = exons[gid]
            seq = genome[chrom]
            n = round(expected_reads(gene_i, group) * rng.uniform(0.85, 1.15))
            for k in range(max(0, n)):
                frag = pick_fragment(seq, start, end, rng)
                name = f"{sample}:{gid}:{total + k}"
                r1_out.write(f"@{name}/1\n{frag[:READLEN]}\n+\n{QUAL}\n")
                r2_out.write(f"@{name}/2\n{revcomp(frag[-READLEN:])}\n+\n{QUAL}\n")
            total += max(0, n)
    print(f"    {sample:8s} ({group:7s}) -> {total:6d} read pairs")

print(">>> demo reads ready in data/")

## Step 5 — Run the alignment pipeline *(bash)*

For each sample: FastQC → Trimmomatic → STAR → featureCounts. The result is a single table of read counts per gene, `counts/counts.txt`.

In [ ]:
%%writefile run_pipeline.sh
#!/usr/bin/env bash
set -euo pipefail

THREADS=2
INDEX=ref/star_index
GTF=ref/genes.gtf

# Locate Trimmomatic's bundled adapter file (its folder name includes a version).
BIN_DIR="$(dirname "$(command -v trimmomatic)")"
ADAPTERS="$(ls "${BIN_DIR}/../share/"trimmomatic*/adapters/TruSeq3-PE.fa 2>/dev/null | head -n1)"

mkdir -p qc trimmed aligned counts
for R1 in data/*_R1.fastq.gz; do
  SAMPLE="$(basename "$R1" _R1.fastq.gz)"
  R2="data/${SAMPLE}_R2.fastq.gz"
  echo "==== ${SAMPLE} ===="

  echo ">>> FastQC"
  fastqc -q -t "$THREADS" -o qc "$R1" "$R2"

  echo ">>> Trimmomatic"
  trimmomatic PE -phred33 -threads "$THREADS" "$R1" "$R2" \
    "trimmed/${SAMPLE}_R1.fq.gz" "trimmed/${SAMPLE}_R1.unpaired.fq.gz" \
    "trimmed/${SAMPLE}_R2.fq.gz" "trimmed/${SAMPLE}_R2.unpaired.fq.gz" \
    "ILLUMINACLIP:${ADAPTERS}:2:30:10" LEADING:3 TRAILING:3 SLIDINGWINDOW:4:20 MINLEN:36

  echo ">>> STAR"
  STAR --runThreadN "$THREADS" --genomeDir "$INDEX" \
    --readFilesIn "trimmed/${SAMPLE}_R1.fq.gz" "trimmed/${SAMPLE}_R2.fq.gz" \
    --readFilesCommand zcat --outSAMtype BAM SortedByCoordinate \
    --outFileNamePrefix "aligned/${SAMPLE}_"
done

echo ">>> featureCounts"
featureCounts -T "$THREADS" -p --countReadPairs -a "$GTF" \
  -o counts/counts.txt aligned/*_Aligned.sortedByCoord.out.bam

echo ">>> counts ready in counts/counts.txt"


In [ ]:
!bash run_pipeline.sh

## Step 6 — Differential expression in R *(inline)*

Now we install R and switch to working **in R, inside the notebook**. `rpy2` gives us the `%%R` magic: every `%%R` cell shares one R session (so variables carry over between cells) and any plot it makes appears inline.

**Install R and turn on the `%%R` magic:**

In [ ]:
# Install R, edgeR (differential expression), and rpy2 — which lets us run R
# inside the notebook with the %%R magic.
!mamba install -y -n base -c conda-forge -c bioconda bioconductor-edger r-base r-ggplot2 r-pheatmap rpy2

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
cat(R.version.string, "\n")   # R now runs inside the notebook

**Load the counts and fit the model** (edgeR, *mip6Δ* vs control):

In [ ]:
%%R
suppressMessages({library(edgeR); library(ggplot2); library(pheatmap)})

# Load the featureCounts matrix from Step 5 (the first 6 columns are annotation).
fc     <- read.delim("counts/counts.txt", comment.char = "#")
counts <- as.matrix(fc[, 7:ncol(fc)])
rownames(counts) <- fc$Geneid
colnames(counts) <- sub("_Aligned.*", "", basename(colnames(counts)))

# The columns come out alphabetically: ctrl_1..3 then mip6_1..3.
group <- factor(c("control","control","control","mip6","mip6","mip6"),
                levels = c("control","mip6"))

y  <- DGEList(counts = counts, group = group)
y  <- y[filterByExpr(y), , keep.lib.sizes = FALSE]
y  <- calcNormFactors(y)
y  <- estimateDisp(y)
et <- exactTest(y)                       # mip6 vs control
res <- topTags(et, n = Inf)$table
res$gene <- rownames(res)

dir.create("figures", showWarnings = FALSE)
write.csv(res, "counts/de_results.csv", row.names = FALSE)
cat("genes tested:", nrow(res), "| significant (FDR < 0.05):", sum(res$FDR < 0.05), "\n")
head(res)

**Volcano plot** — shown inline and saved to `figures/`:

In [ ]:
%%R -w 700 -h 500 -u px
res$sig <- with(res, ifelse(FDR < 0.05 & abs(logFC) > 1,
                     ifelse(logFC > 0, "up", "down"), "ns"))
p <- ggplot(res, aes(logFC, -log10(FDR), colour = sig)) +
  geom_point(size = 1, alpha = 0.7) +
  scale_colour_manual(values = c(up = "#b2182b", down = "#2166ac", ns = "grey70")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed") +
  labs(title = "mip6Δ vs control",
       x = "log2 fold change", y = "-log10 FDR", colour = NULL) +
  theme_bw()
print(p)                                 # show the plot inline
ggsave("figures/volcano.png", p, width = 7, height = 5, dpi = 150)

**Heatmap** of the top differentially-expressed genes:

In [ ]:
%%R -w 700 -h 800 -u px
top    <- head(rownames(res[order(res$FDR), ]), 40)
logcpm <- cpm(y, log = TRUE)[top, ]
ann    <- data.frame(group = group)
rownames(ann) <- colnames(logcpm)
heat <- pheatmap(logcpm, scale = "row", annotation_col = ann,
                 show_rownames = FALSE, fontsize = 8)
ggsave("figures/heatmap.png", heat$gtable, width = 7, height = 8, dpi = 150)

## Step 7 — Results table *(python)*

The R step saved `counts/de_results.csv`. Here it is back on the Python side, sorted by significance.

In [ ]:
import pandas as pd
de = pd.read_csv('counts/de_results.csv')
de.sort_values('FDR').head(10)

## Step 8 — Download your results

Colab is temporary, so save your output before the session ends.

In [ ]:
from google.colab import files
!zip -qr results.zip counts figures
files.download('results.zip')